In [118]:
import numpy as np
import os
import scipy.optimize as opt

In [119]:
def logsumexp(x: np.ndarray) -> float:
    """Compute log(sum(exp(x))) such that max(x) is used to avoid overflow"""
    M = np.max(x)
    return M + np.log(np.sum(np.exp(x - M)))

In [120]:
def letter_to_index(letter: str) -> int:
    """Convert a letter to an index (0 for 'a', 1 for 'b', ..., 25 for 'z')."""
    return ord(letter) - ord('a')

1. Load model.txt

W = 26 * 128

T = 26 * 26

In [121]:
def load_model(path, K, d):
    """Model parameters are stored in a text file as a single vector. 
    The first K*d entries correspond to W, and the next K*K entries correspond to T."""
    theta = np.loadtxt(path, dtype=np.float64) 

    # if K*d + K*K != theta.size:
    #     raise ValueError(f"Bad model length {theta.size} for K={K}")

    W = theta[:K*d].reshape(K, d)
    T = theta[K*d:].reshape(K, K)
    return W, T

2. Load train.txt

In [122]:
def load_data(path):
    sequences = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            parts = line.split()
            label_char = parts[1]
            feats = [float(v) for v in parts[5:]]

            X = np.array([feats], dtype=np.float64)         # (1, d)
            y = np.array([letter_to_index(label_char)], dtype=np.int64)  # (1,)
            sequences.append((X, y))

    return sequences

3. Write score function

In [123]:
def score_function(X, y, W, T):
    """Compute the score of a sequence (X,y) given model parameters W and T"""
    score = 0.0
    n = X.shape[0]
    for s in range(n-1):
        score += W[y[s]].dot(X[s]) + T[y[s], y[s+1]]

    score += W[y[n-1]].dot(X[n-1])

    return score

4. Define partition function partition function Z

In [124]:
def compute_Z_forward(X, W, T):
    """Compute the partition function Z(X) for a sequence X given model parameters W and T"""
    
    # Using dynamic progrmming to compute the partition function Z(X)
    n, d = X.shape
    K, dW = W.shape

    alpha = np.zeros((n, K), dtype = np.float64)

    # Pre-compute the A matrix which is the score of each letter at each position without transition scores
    A = X @ W.T
    alpha[0, :] = A[0, :]

    """Iterate through the sequence and compute the alpha values using the A matrix and transition scores T.
    This is the forward pass for computing the partition function Z(X)"""
    for s in range(1, n):
        for j in range(K):
            prev_terms = alpha[s-1, :] + T[:, j]
            alpha[s, j] = logsumexp(prev_terms) + A[s, j]
        
    logZ = logsumexp(alpha[n-1, :])
    
    return logZ, alpha

5. backward pass

In [125]:
def viterbi_decode_WTX(X, W, T):
    """
    Backward DP in log-space for CRF.
    Returns beta of shape (n, K).
    beta[s,i] = log sum of all suffix path scores from position s+1..n-1
                given y_s = i
    """

    n, d = X.shape
    K = W.shape[0]

    # Emission scores: A[s, j] = W[j] · X[s]
    A = X @ W.T  # (n, K)

    beta = np.zeros((n, K), dtype=np.float64)  # beta[n-1,:] = 0

    for s in range(n - 2, -1, -1):
        # for each current label i, sum over next label j
        # beta[s,i] = logsumexp_j( T[i,j] + A[s+1,j] + beta[s+1,j] )
        beta[s, :] = logsumexp(T + (A[s + 1, :] + beta[s + 1, :])[None, :], axis=1)
    
    delta = np.zeros((n, K), dtype=np.float64)
    DP = np.zeros((n, K), dtype=np.int32)

    delta[0, :] = A[0, :]
    for s in range(1, n):
        scores = delta[s - 1][:, None] + T  # (K, K): prev i -> curr j
        DP[s, :] = np.argmax(scores, axis=0)
        delta[s, :] = np.max(scores, axis=0) + A[s, :]

    y_pred = np.zeros(n, dtype=np.int32)
    y_pred[n - 1] = np.argmax(delta[n - 1, :])
    for s in range(n - 2, -1, -1):
        y_pred[s] = DP[s + 1, y_pred[s + 1]]

    return beta, y_pred

6. Compute node marginals

In [126]:
def compute_node_marginals(alpha, beta, logZ):
    """Compute the node marginals p(y_s | X) for each position s given alpha, beta, and logZ"""
    n, K = alpha.shape
    node_marginals = np.zeros((n, K), dtype=np.float64)

    # node_marginals measure the probability of each label at each position given the sequence X, and are computed using the alpha and beta values from the forward and backward passes, normalized by the partition function Z(X).
    for s in range(n):
        log_prob = alpha[s, :] + beta[s, :] - logZ
        
        """ probs are the unnormalized probabilities of each label at position s, computed by exponentiating the log probabilities. 
        To avoid numerical instability, we subtract the maximum log probability m from log_prob before exponentiating, 
        which ensures that we are working with smaller numbers and prevents overflow."""
        m = np.max(log_prob)
        probs = np.exp(log_prob - m)
        node_marginals[s, :] = probs / np.sum(probs)

    return node_marginals

7. Compute edge marginals

In [127]:
def compute_edge_marginals(X, W, T, alpha, beta, logZ):
    """Compute the edge marginals p(y_s, y_{s+1} | X) for each position s given X, W, T, alpha, beta, and logZ"""
    n, d = X.shape
    K = W.shape[0]
    edge_marginals = np.zeros((n-1, K, K), dtype=np.float64)

    # edge_marginals measure the joint probability of pairs of labels at adjacent positions given the sequence X, and are computed using the alpha and beta values from the forward and backward passes, along with the model parameters W and T, normalized by the partition function Z(X).
    A = X @ W.T

    # log_probs are the unnormalized log probabilities of each pair of labels at positions s and s+1, computed using the alpha values at position s, the transition scores T, the A matrix for the scores of each label at each position, and the beta values at position s+1.
    for s in range(n-1):
        log_probs = (alpha[s][:, None]
                     + T
                     + A[s+1][None, :]
                     + beta[s+1][None, :]
                     - logZ)

        # To avoid numerical overflow, subtract the maximum log probability from each log probability before exponentiating
        m = np.max(log_probs)
        probs = np.exp(log_probs - m)
        edge_marginals[s] = probs / np.sum(probs)

    return edge_marginals

8. Compute log-likelihood of one sequence P(X,y)

In [128]:
def compute_logp_y_given_X(X, y, W, T):

    # Compute the log probability of a sequence (X,y) given model parameters W and T using the score function and the partition function Z(X)
    logZ, _ = compute_Z_forward(X, W, T)
    score = score_function(X, y, W, T)
    return score - logZ

9. Differentiate log likelihood, find gradient

In [129]:
def gradient_loglikelihood(X, y, W, T):
    """Compute the gradient of the log likelihood with respect to W and T for a single sequence (X,y)"""
    n, d = X.shape
    K = W.shape[0]

    # Compute the node and edge marginals
    logZ, alpha = compute_Z_forward(X, W, T)
    beta, y_pred = viterbi_decode_WTX(X, W, T)
    node_marginals = compute_node_marginals(alpha, beta, logZ)
    edge_marginals = compute_edge_marginals(X, W, T, alpha, beta, logZ)

    # Initialize gradients
    grad_W = np.zeros_like(W)
    grad_T = np.zeros_like(T)

    # Compute the gradient with respect to W
    for s in range(n):
        grad_W[y[s]] += X[s]  
        grad_W -= node_marginals[s][:, None] * X[s]

    # Compute the gradient with respect to T
    for s in range(n-1):
        grad_T[y[s], y[s+1]] += 1
        grad_T -= edge_marginals[s]

    return grad_W, grad_T, y_pred

10. optimize the parameters W and T

In [130]:
def optimize_parameters(sequences, W, T):
    """Optimize the parameters W and T using gradient ascent on the log likelihood for a list of sequences"""
    total_log_p = 0
    total_gradW = 0
    total_gradT = 0

    y_hat = []

    for X, y in sequences:

        lop_p = compute_logp_y_given_X(X, y, W, T)
        grad_W, grad_T, y_pred = gradient_loglikelihood(X, y, W, T)
        total_log_p += lop_p
        total_gradW += grad_W
        total_gradT += grad_T
        y_hat.append(y_pred)

    return total_log_p, total_gradW, total_gradT

    

11. write predictions

In [131]:
def write_predictions(path, gradW, gradT, sequences):
    g = np.concatenate([gradW.reshape(-1, order="C"),
                    gradT.reshape(-1, order="C")])
    np.savetxt(path, g, fmt="%.15f")

12. Average over the gradients of the data

In [ ]:
def main():
    current_dir = os.getcwd()

    # Go up one level to project_root
    project_root = os.path.dirname(current_dir)

    data_path = os.path.join(project_root, "data", "train.txt")
    data = load_data(data_path)

    K = 26
    d = data[0][0].shape[1]

    model_path = os.path.join(project_root, "data", "model.txt")
    W, T = load_model(model_path, K, d)

    y_hat = []

    total_logp = 0.0
    total_gradW = np.zeros_like(W)
    total_gradT = np.zeros_like(T)

    for X, y in data:
        lop_p = compute_logp_y_given_X(X, y, W, T)
        grad_W, grad_T, y_pred = gradient_loglikelihood(X, y, W, T)
        total_logp += lop_p
        total_gradW += grad_W
        total_gradT += grad_T
        y_hat.append(y_pred)

    n = len(data)
    avg_logp = total_logp / n
    avg_gradW = total_gradW / n
    avg_gradT = total_gradT / n
    write_predictions("../result/gradient.txt", avg_gradW, avg_gradT, data)
    print("avg_log_likelihood: ", avg_logp)

if __name__ == "__main__":
    main()

avg_log_likelihood:  -4.140274439213303


# Test